# Fig 5f–g: GSEA of TIL zones

**Produces**
- **Fig 5f**: NES of the preranked GSEA between TIL-high and TIL-low sections within each TIL zone
  (TIL+, BZ1, BZ2); top 5 gene sets per direction (FDR < 0.05) from Hallmark, BioCarta and KEGG
  Medicus, plus the best significant TGFβ set if it missed the top 5 (`lollipop_between_group`).
- **Fig 5g**: the same plot for the between-zone comparisons (TIL+ vs BZ1, BZ1 vs BZ2) within
  TIL-high and within TIL-low sections (`lollipop_TIL+_vs_BZ1`, `lollipop_BZ1_vs_BZ2`).

**Origin**: Spencer Watson, EFS `Analysis_Repo/Notebooks/TIL_Niche_Comparisons_v2_clean.ipynb`,
cell 63 ("GSEA lollipop plots v4") with its markdown (cell 62). It reads the frozen table written by
cell 61 of the same notebook, `results/visium/TIL_zone_prerank_GSEA_unbiased/gsea_all_comparisons_combined.csv`
(gseapy prerank, 1,000 permutations; Welch t of per-section mean log-normalised expression between
groups, paired t between zones; 57 TIL-high and 148 TIL-low sections with TIL+ spots). Cell 61 needs
the whole log-normalised Visium matrix and is not re-run.

**Compared with the published outputs**: the frozen table is identical to Supp Table 5b
(`supplementary_tables/5b_gsea_all_comparisons_combined.csv`), and the three PNGs are byte-identical
to those of the source run in `GSEA_OUTDIR` (both checked below).

In [ ]:
import filecmp
import shutil

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gbmcollaboration.paper import config, output

In [ ]:
# Parameters
OUT = output.setup("fig5fg_TIL_zone_GSEA")
plt.rcParams["savefig.transparent"] = False
GSEA_OUTDIR = config.EFS / "results/visium/TIL_zone_prerank_GSEA_unbiased"   # frozen gseapy prerank results
SUPP_TABLE_5B = config.EFS / "supplementary_tables/5b_gsea_all_comparisons_combined.csv"
PANELS = {"lollipop_between_group": "Fig 5f", "lollipop_TIL+_vs_BZ1": "Fig 5g TIL+ vs BZ1",
          "lollipop_BZ1_vs_BZ2": "Fig 5g BZ1 vs BZ2"}   # also written to OUT_ROOT/panels/

# =====================================================================
# CONFIG
# =====================================================================
RESULTS_CSV = GSEA_OUTDIR / "gsea_all_comparisons_combined.csv"
FDR_THRESH  = 0.05
TOP_N       = 5
LIBRARIES   = ["HALLMARK", "BIOCARTA", "KEGG",]  # "REACTOME"

# ── Forced gene-set patterns ────────────────────────────────────────
# Case-insensitive substring match. If a significant gene set matches
# any pattern but did not make top N, it is appended as position N+1.
FORCE_PATTERNS = ["TGFB"]

# ── Exclusion patterns ──────────────────────────────────────────────
# Case-insensitive substring matching on gene_set name.
# Populate after reviewing the printed gene set lists below.
EXCLUDE_PATTERNS = [
    "RUNX1 Interacts With Co-Factors Whose Precise Effect On RUNX1 Targets Is Not Known R-HSA-8939243",
    "APC/C:Cdh1 Mediated Degradation Of Cdc20 And APC/C:Cdh1 Targets In Late Mitosis/Early G1 R-HSA-174178",
    "Response Of Mtb To Phagocytosis R-HSA-963769","Erythropoietin Activates RAS R-HSA-9027284",
    "Creation Of C4 And C2 Activators R-HSA-166786","Cdc20:Phospho-APC/C Mediated Degradation Of Cyclin A R-HSA-17418",
    "Transferrin Endocytosis And Recycling R-HSA-917977","Infection With Mycobacterium Tuberculosis R-HSA-9635486"
]

COMP_LABELS = {
    "TIL-High_vs_TIL-Low_in_TIL+": "TIL+",
    "TIL-High_vs_TIL-Low_in_BZ1":  "BZ1",
    "TIL-High_vs_TIL-Low_in_BZ2":  "BZ2",
    "TIL+_vs_BZ1_in_TIL-High":     "TIL-High",
    "TIL+_vs_BZ1_in_TIL-Low":      "TIL-Low",
    "BZ1_vs_BZ2_in_TIL-High":      "TIL-High",
    "BZ1_vs_BZ2_in_TIL-Low":       "TIL-Low",
}

ZONE_COLORS = {
    "TIL+":     "#8B0000",
    "BZ1":      "#FF8C00",
    "BZ2":      "#FFD300",
    "TIL-High": "#D55E00",
    "TIL-Low":  "#0072B2",
}

BLOCK_GAP = 1.0

print(f"Frozen GSEA table identical to Supp Table 5b: "
      f"{filecmp.cmp(GSEA_OUTDIR / 'gsea_all_comparisons_combined.csv', SUPP_TABLE_5B, shallow=False)}")

In [ ]:
# =====================================================================
# HELPERS
# =====================================================================
def get_library(gs):
    gs = str(gs).upper()
    if gs.startswith("HALLMARK"):                            return "HALLMARK"
    if gs.startswith("BIOCARTA"):                            return "BIOCARTA"
    if "R-HSA-" in gs:                                       return "REACTOME"
    if gs.startswith("KEGG") or gs.startswith("MEDICUS"):    return "KEGG"
    if gs.startswith("PID"):                                 return "PID"
    if gs.startswith("GOBP") or gs.startswith("GO_"):        return "GOBP"
    if gs.startswith("GSE"):                                 return "IMMUNESIGDB"
    return "OTHER"


def is_excluded(gs):
    if not EXCLUDE_PATTERNS:
        return False
    gs_upper = str(gs).upper()
    return any(pat.upper() in gs_upper for pat in EXCLUDE_PATTERNS)


def _has_force_pattern(gs):
    """Check if a gene set name matches any FORCE_PATTERNS (case-insensitive)."""
    gs_upper = str(gs).upper()
    return any(pat.upper() in gs_upper for pat in FORCE_PATTERNS)


def select_top(df_sub, top_n=TOP_N, fdr_thresh=FDR_THRESH):
    sig = df_sub[df_sub["FDR"] < fdr_thresh].copy()
    pos = sig[sig["NES"] > 0].nlargest(top_n, "NES")
    neg = sig[sig["NES"] < 0].nsmallest(top_n, "NES")

    # ── Force TGFB gene sets into last position if not already present ──
    already = set(pos["gene_set"]).union(set(neg["gene_set"]))
    sig_force = sig[
        sig["gene_set"].apply(_has_force_pattern)
        & ~sig["gene_set"].isin(already)
    ]

    if not sig_force.empty:
        # Best positive TGFB hit (highest NES)
        force_pos = sig_force[sig_force["NES"] > 0]
        if not force_pos.empty:
            best_pos = force_pos.nlargest(1, "NES")
            pos = pd.concat([pos, best_pos], ignore_index=True)

        # Best negative TGFB hit (lowest NES)
        force_neg = sig_force[sig_force["NES"] < 0]
        if not force_neg.empty:
            best_neg = force_neg.nsmallest(1, "NES")
            neg = pd.concat([neg, best_neg], ignore_index=True)

    return pd.concat([pos, neg], ignore_index=True)


# =====================================================================
# SHARED PLOTTING FUNCTION
# =====================================================================
def draw_lollipop(blocks, comp_order, label_map, color_map,
                  title, direction_labels, save_stem):
    """
    Draw and save a single lollipop figure.

    blocks          : dict  comp_name -> DataFrame of selected rows
    comp_order      : list of comp_names in display order (bottom -> top)
    label_map       : comp_name -> display label
    color_map       : display_label -> hex color
    title           : str
    direction_labels: (neg_label, pos_label)
    save_stem       : str  filename stem (no extension)
    """
    # ── Build y-axis layout ──
    y_labels = []
    y_pos_map = {}
    y_comps = []
    y_gs = []
    y_current = 0

    for comp in comp_order:
        block = blocks.get(comp, pd.DataFrame())
        if block.empty:
            continue
        block = block.sort_values("NES", ascending=True)
        for _, row in block.iterrows():
            gs = row["gene_set"]
            y_pos_map[(comp, gs)] = y_current
            y_labels.append(str(gs))
            y_comps.append(comp)
            y_gs.append(gs)
            y_current += 1
        y_current += BLOCK_GAP

    n_rows = len(y_labels)
    if n_rows == 0:
        print(f"  {save_stem}: no significant gene sets — skipping plot")
        return

    # ── Figure size from content ──
    row_height = 0.36
    fig_height = n_rows * row_height + (len(blocks) - 1) * BLOCK_GAP * row_height + 1.5
    fig, ax = plt.subplots(figsize=(9, fig_height))

    # ── Draw lollipops ──
    for comp in comp_order:
        block = blocks.get(comp, pd.DataFrame())
        label = label_map.get(comp, comp)
        color = color_map.get(label, "gray")
        for _, row in block.iterrows():
            gs = row["gene_set"]
            y = y_pos_map.get((comp, gs))
            if y is None:
                continue
            nes = row["NES"]
            sig = row["FDR"] < FDR_THRESH
            ax.plot([0, nes], [y, y], color=color, lw=2, alpha=0.8)
            ax.scatter(nes, y, color=color,
                       s=60 if sig else 30,
                       edgecolor="black" if sig else "none",
                       linewidth=0.8 if sig else 0,
                       zorder=5, alpha=1.0 if sig else 0.5)

    # ── Block separators ──
    prev_comp = None
    for i, comp in enumerate(y_comps):
        if prev_comp is not None and comp != prev_comp:
            y_above = y_pos_map[(prev_comp, y_gs[i - 1])]
            y_below = y_pos_map[(comp, y_gs[i])]
            ax.axhline((y_above + y_below) / 2, color="gray",
                       ls="--", lw=0.5, alpha=0.4)
        prev_comp = comp

    # ── Y-axis: tight to data ──
    all_ys = sorted(y_pos_map.values())
    ax.set_yticks(all_ys)
    ax.set_yticklabels(y_labels, fontsize=10)
    ax.set_ylim(min(all_ys) - 0.8, max(all_ys) + 0.8)

    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("NES", fontsize=10)
    ax.set_title(title, fontsize=10, fontweight="bold")

    # ── Expand right margin for block labels ──
    xlim = ax.get_xlim()
    ax.set_xlim(xlim[0], xlim[1] + abs(xlim[1] - xlim[0]) * 0.20)
    xlim = ax.get_xlim()

    for comp in comp_order:
        block = blocks.get(comp, pd.DataFrame())
        block_ys = [y_pos_map[(comp, gs)] for gs in block["gene_set"]
                    if (comp, gs) in y_pos_map]
        if not block_ys:
            continue
        mid_y = np.mean(block_ys)
        label = label_map.get(comp, comp)
        color = color_map.get(label, "gray")
        ax.text(xlim[1] * 0.98, mid_y, label,
                va="center", ha="right", fontsize=10, fontweight="bold",
                color=color)

    # ── Directional annotation ──
    neg_lbl, pos_lbl = direction_labels
    ax.text(0.5, -0.06,
            f"← {neg_lbl} enriched          {pos_lbl} enriched →",
            ha="center", fontsize=8, fontstyle="italic", color="gray",
            transform=ax.transAxes)

    # ── Clean spines, no grid ──
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    # ── Legend ──
    # Unique labels in order
    seen = []
    legend_handles = []
    for comp in comp_order:
        label = label_map.get(comp, comp)
        if label not in seen:
            seen.append(label)
            legend_handles.append(
                mpatches.Patch(color=color_map.get(label, "gray"),
                               label=label)
            )
    legend_handles.extend([
        plt.scatter([], [], s=60, c="gray", edgecolor="black",
                    linewidth=0.8, label="FDR < 0.05"),
        plt.scatter([], [], s=30, c="gray", edgecolor="none",
                    alpha=0.5, label="FDR ≥ 0.05"),
    ])
    ax.legend(handles=legend_handles, loc="lower right",
              fontsize=12, framealpha=0.8)

    # ── Save ──
    fig.savefig(OUT / f"{save_stem}.png",
                dpi=200, bbox_inches="tight")
    fig.savefig(OUT / f"{save_stem}.pdf",
                bbox_inches="tight")
    output.PANELS_DIR.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        shutil.copy(OUT / f"{save_stem}.{ext}", output.PANELS_DIR / f"{output.panel_name(PANELS[save_stem])}.{ext}")
    pd.concat(blocks, names=["comparison"]).to_csv(OUT / f"{save_stem}.csv")
    shutil.copy(OUT / f"{save_stem}.csv", output.PANELS_DIR / f"{output.panel_name(PANELS[save_stem])}.csv")
    plt.show()
    plt.close(fig)
    print(f"  Saved {save_stem}.png/pdf")

### Plot GSEA Results

In [ ]:
# GSEA lollipop plots v4: between-group and between-zone comparisons
# ─────────────────────────────────────────────────────────────────────
# Libraries: Hallmark + Biocarta + KEGG Medicus  (Reactome excluded)
# Optional EXCLUDE_PATTERNS filter (empty by default — user populates)
# Each zone×direction block gets up to TOP_N gene sets (FDR < threshold)
# + forced TGFB gene sets appended as position TOP_N+1 if significant
#   but not already in top N
# Blocks with < TOP_N or 0 hits shown as-is
#
# Outputs:
#   1) lollipop_between_group.png/pdf    (3 zone blocks)
#   2) lollipop_TIL+_vs_BZ1.png/pdf     (TIL-High / TIL-Low blocks)
#   3) lollipop_BZ1_vs_BZ2.png/pdf      (TIL-High / TIL-Low blocks)
# ─────────────────────────────────────────────────────────────────────

# =====================================================================
# LOAD & FILTER
# =====================================================================
df = pd.read_csv(RESULTS_CSV, low_memory=False)
df["NES"] = df["NES"].astype(float)
df["FDR"] = df["FDR"].astype(float)

if "library" not in df.columns:
    df["library"] = df["gene_set"].apply(get_library)

df_filt = df[df["library"].isin(LIBRARIES)].copy()

n_before = len(df_filt)
df_filt = df_filt[~df_filt["gene_set"].apply(is_excluded)].copy()
n_excluded = n_before - len(df_filt)
print(f"Loaded {len(df)} rows -> filtered to {LIBRARIES}: {n_before} rows")
if n_excluded:
    print(f"Excluded {n_excluded} gene sets matching EXCLUDE_PATTERNS "
          f"-> {len(df_filt)} rows")


# =====================================================================
# PLOT 1: Between-group (TIL-High vs TIL-Low within each zone)
# =====================================================================
print("\n── Between-group ──")
bg_order = [
    "TIL-High_vs_TIL-Low_in_TIL+",
    "TIL-High_vs_TIL-Low_in_BZ1",
    "TIL-High_vs_TIL-Low_in_BZ2",
]
bg_comps = [c for c in bg_order if c in df_filt["comparison"].unique()]

bg_blocks = {}
for comp in bg_comps:
    sub = df_filt[df_filt["comparison"] == comp]
    top = select_top(sub)
    bg_blocks[comp] = top
    zone = COMP_LABELS.get(comp, comp)
    n_pos = (top["NES"] > 0).sum()
    n_neg = (top["NES"] < 0).sum()
    # Report forced TGFB sets
    n_forced = top["gene_set"].apply(_has_force_pattern).sum()
    forced_note = f" (incl. {n_forced} forced TGFB)" if n_forced else ""
    print(f"  {zone}: {n_pos} TIL-High + {n_neg} TIL-Low enriched{forced_note}")

draw_lollipop(
    blocks=bg_blocks,
    comp_order=bg_comps,
    label_map=COMP_LABELS,
    color_map=ZONE_COLORS,
    title=(
        "Between-group GSEA: TIL-High vs TIL-Low by zone\n"
        f"(Hallmark + Biocarta + KEGG, "
        f"top {TOP_N}/direction, FDR < {FDR_THRESH})"
    ),
    direction_labels=("TIL-Low", "TIL-High"),
    save_stem="lollipop_between_group",
)


# =====================================================================
# PLOTS 2 & 3: Between-zone (separate plot per zone pair)
# =====================================================================
bz_pairs = [
    ("TIL+ vs BZ1",
     ["TIL+_vs_BZ1_in_TIL-High", "TIL+_vs_BZ1_in_TIL-Low"],
     "lollipop_TIL+_vs_BZ1"),
    ("BZ1 vs BZ2",
     ["BZ1_vs_BZ2_in_TIL-High", "BZ1_vs_BZ2_in_TIL-Low"],
     "lollipop_BZ1_vs_BZ2"),
]

all_bz_blocks = {}

for pair_title, comps, save_stem in bz_pairs:
    print(f"\n── {pair_title} ──")
    pair_blocks = {}
    for comp in comps:
        if comp not in df_filt["comparison"].unique():
            pair_blocks[comp] = pd.DataFrame()
            continue
        sub = df_filt[df_filt["comparison"] == comp]
        top = select_top(sub)
        pair_blocks[comp] = top
        all_bz_blocks[comp] = top
        grp = "TIL-High" if "TIL-High" in comp else "TIL-Low"
        n_pos = (top["NES"] > 0).sum()
        n_neg = (top["NES"] < 0).sum()
        n_forced = top["gene_set"].apply(_has_force_pattern).sum()
        forced_note = f" (incl. {n_forced} forced TGFB)" if n_forced else ""
        print(f"  {grp}: {n_pos} proximal + {n_neg} distal enriched{forced_note}")

    comps_present = [c for c in comps
                     if c in df_filt["comparison"].unique()]

    pair_label_map = {}
    for c in comps_present:
        grp = "TIL-High" if "TIL-High" in c else "TIL-Low"
        pair_label_map[c] = grp

    zone_a, zone_b = pair_title.split(" vs ")

    draw_lollipop(
        blocks=pair_blocks,
        comp_order=comps_present,
        label_map=pair_label_map,
        color_map=ZONE_COLORS,
        title=(
            f"Between-zone GSEA: {pair_title}\n"
            f"(Hallmark + Biocarta + KEGG, "
            f"top {TOP_N}/direction, FDR < {FDR_THRESH})"
        ),
        direction_labels=(zone_b, zone_a),
        save_stem=save_stem,
    )


# =====================================================================
# PRINT ALL GENE SETS IN EACH PLOT (for curation / exclusion list)
# =====================================================================
print("\n" + "=" * 80)
print("BETWEEN-GROUP — ALL GENE SETS IN PLOT")
print("Copy unwanted names into EXCLUDE_PATTERNS to remove them.")
print("=" * 80)
for comp in bg_comps:
    block = bg_blocks[comp]
    zone = COMP_LABELS.get(comp, comp)
    print(f"\n  {zone} ({len(block)} gene sets):")
    if block.empty:
        print("    (none significant)")
    for _, r in block.sort_values("NES", ascending=False).iterrows():
        direction = "TIL-High" if r["NES"] > 0 else "TIL-Low"
        sig_mark = "***" if r["FDR"] < FDR_THRESH else ""
        forced = " [FORCED]" if _has_force_pattern(r["gene_set"]) and \
                 r["gene_set"] not in set(
                     df_filt[df_filt["comparison"] == comp]
                     .nlargest(TOP_N, "NES")["gene_set"]
                 ) else ""
        print(f"    {r['gene_set']:70s} "
              f"NES={r['NES']:+.3f}  FDR={r['FDR']:.4f}  "
              f"[{direction}] {sig_mark}{forced}")

print("\n" + "=" * 80)
print("BETWEEN-ZONE — ALL GENE SETS IN PLOTS")
print("Copy unwanted names into EXCLUDE_PATTERNS to remove them.")
print("=" * 80)
for pair_title, comps, save_stem in bz_pairs:
    print(f"\n  {pair_title}:")
    zone_a, zone_b = pair_title.split(" vs ")
    for comp in comps:
        grp = "TIL-High" if "TIL-High" in comp else "TIL-Low"
        block = all_bz_blocks.get(comp, pd.DataFrame())
        print(f"    [{grp}] ({len(block)} gene sets):")
        if block.empty:
            print("      (none significant)")
            continue
        for _, r in block.sort_values("NES", ascending=False).iterrows():
            direction = zone_a if r["NES"] > 0 else zone_b
            sig_mark = "***" if r["FDR"] < FDR_THRESH else ""
            forced = " [FORCED]" if _has_force_pattern(r["gene_set"]) and \
                     r["gene_set"] not in set(
                         df_filt[df_filt["comparison"] == comp]
                         .nlargest(TOP_N, "NES")["gene_set"]
                     ) else ""
            print(f"      {r['gene_set']:70s} "
                  f"NES={r['NES']:+.3f}  FDR={r['FDR']:.4f}  "
                  f"[{direction}] {sig_mark}{forced}")

In [ ]:
# Figures compared with the published PNGs of the source run (in GSEA_OUTDIR)
for stem in PANELS:
    a, b = plt.imread(OUT / f"{stem}.png"), plt.imread(GSEA_OUTDIR / f"{stem}.png")
    same_shape = a.shape == b.shape
    n_diff = int((np.abs(a - b).max(axis=-1) > 0).sum()) if same_shape else None
    print(f"{stem}: byte-identical {filecmp.cmp(OUT / f'{stem}.png', GSEA_OUTDIR / f'{stem}.png', shallow=False)}; "
          f"size {a.shape[:2]} vs published {b.shape[:2]}; differing pixels {n_diff}")

In [ ]:
print(output.versions(["numpy", "pandas", "matplotlib"]).to_string(index=False))